In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy import signal
from numpy import linalg as LA
import os


### 実験設定


In [2]:
PARTICIPANTS = ['oba', 'ono', 'pon', 'kuno', 'john', 'konan', 
                'obara', 'fukuzawa', 'kiuchi', 'yanaze', 'adachi', 'iwasaki']
MASSES = [60.9, 63.8, 68.7, 65.9, 77, 74, 63.8, 64.5, 73.2, 53.5, 70.7, 47.9]
CONDITIONS = ['h', 'm', 'l']

# サンプリング周波数
DEVICE_FREQ = 100
MOCAP_FREQ = 250
FORCE_FREQ = 1000

# フィルタ設定
CUTOFF_FREQ = 6
FILTER_ORDER = 4

### 1. データ読み込み関数

In [3]:
def load_participant_data(participant, condition, mass):
    """
    指定された実験協力者・条件のデータを読み込む
    
    Parameters:
        participant: 実験協力者名
        condition: 実験条件 ('h', 'm', 'l')
        mass: 体重 [kg]
    
    Returns:
        dict: 各種データフレームと体重情報
    """
    print(f"\n{'='*60}")
    print(f"Processing: {participant} - Condition: {condition.upper()} (Mass: {mass} kg)")
    print(f"{'='*60}")
    
    # ファイルパス生成
    file_path_left = f"WearableDevices/{participant}_{condition}_left_foot_data.csv"
    file_path_right = f"WearableDevices/{participant}_{condition}_right_foot_data.csv"
    file_path_mocap = f"MotionCaptures/{participant}_{condition}_mocap.csv"
    file_path_force = f"3DGroundForces/{participant}_{condition}_force.csv"
    
    # データ読み込み
    df_left = pd.read_csv(file_path_left, header=0)
    df_right = pd.read_csv(file_path_right, header=0)
    df_mocap = pd.read_csv(file_path_mocap, header=[2, 5, 6])
    df_force = pd.read_csv(file_path_force, header=10, encoding='shift_jis')
    
    return {
        'left': df_left,
        'right': df_right,
        'mocap': df_mocap,
        'force': df_force,
        'mass': mass,
        'participant': participant,
        'condition': condition
    }

### 2. モーションキャプチャデータの列名整理

In [4]:
def clean_mocap_columns(df_mocap):
    """モーションキャプチャデータの列名を整理"""
    new_columns = []
    
    for col in df_mocap.columns:
        if col[2] == 'Frame':
            new_columns.append('Frame')
        elif 'Time' in col[2]:
            new_columns.append('Time (Seconds)')
        else:
            body_num = col[0].replace('Rigid Body', '').strip()
            name = f"{body_num}_{col[1]}_{col[2]}"
            new_columns.append(name)
    
    df_mocap.columns = new_columns
    return df_mocap

### 3. 床反力データの列名整理

In [5]:
def clean_force_columns(df_force):
    """床反力データの列名を英語に変換"""
    columns_mapping = {
        'Unnamed: 0': 'Time (Seconds)',
        '右-Fx': 'Right_Fx', '右-Fy': 'Right_Fy', '右-Fz': 'Right_Fz',
        '右-Mx': 'Right_Mx', '右-My': 'Right_My', '右-Mz': 'Right_Mz',
        '右-COPx': 'Right_COPx', '右-COPy': 'Right_COPy',
        '左-Fx': 'Left_Fx', '左-Fy': 'Left_Fy', '左-Fz': 'Left_Fz',
        '左-Mx': 'Left_Mx', '左-My': 'Left_My', '左-Mz': 'Left_Mz',
        '左-COPx': 'Left_COPx', '左-COPy': 'Left_COPy',
    }
    return df_force.rename(columns=columns_mapping)

 ### 4. リサンプリング


In [6]:
def process_resampling(df_input, sampling_interval=10):
    """
    データのリサンプリング，軸反転，単位変換を実行
    
    Parameters:
        df_input: 入力データフレーム
        sampling_interval: サンプリング間隔 [ms]
    """
    df = df_input.copy()
    
    # 欠損値補完
    exclude_cols = ['Marker']
    target_cols = df.columns.difference(exclude_cols)
    df[target_cols] = df[target_cols].interpolate(method='linear', axis=0)
    
    # 新しい時間軸作成
    time_min = 0
    time_max = df['ElapsedTime'].max()
    new_time = np.arange(time_min, time_max, sampling_interval)
    df_resampled = pd.DataFrame({'ElapsedTime': new_time})
    
    # マーカー列の処理
    if exclude_cols:
        df_markers = df[['ElapsedTime'] + exclude_cols].dropna(subset=exclude_cols, how='all').copy()
        df_markers['ElapsedTime_rounded'] = (df_markers['ElapsedTime'] / sampling_interval).round() * sampling_interval
        df_markers = df_markers.drop_duplicates(subset=['ElapsedTime_rounded'])
        
        df_resampled['MergeKey'] = df_resampled['ElapsedTime'].round().astype(int)
        df_markers['MergeKey'] = df_markers['ElapsedTime_rounded'].round().astype(int)
        
        df_resampled = pd.merge(df_resampled, df_markers[['MergeKey'] + exclude_cols],
                                on='MergeKey', how='left')
        df_resampled = df_resampled.drop(columns=['MergeKey'])
    
    # 連続値データの補間
    columns_to_exclude = ['ElapsedTime'] + exclude_cols
    for column in df.columns:
        if column in columns_to_exclude:
            continue
        interpolator = interp1d(df['ElapsedTime'], df[column], 
                                kind='linear', fill_value='extrapolate')
        df_resampled[column] = interpolator(new_time)
    
    # 単位変換
    df_resampled['ElapsedTime'] = df_resampled['ElapsedTime'] / 1000
    df_resampled = df_resampled.rename(columns={'ElapsedTime': 'Time (Seconds)'})
    
    
    return df_resampled

### 5. ローパスフィルタ適用

In [7]:
def apply_lowpass_filter(data, cutoff, fs, order=4):
    """Butterworthローパスフィルタ"""
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    return signal.filtfilt(b, a, data, axis=0)

def process_smoothing_dataframe(df_input, fs=DEVICE_FREQ, cutoff=CUTOFF_FREQ, order=FILTER_ORDER):
    """データフレーム全体にフィルタを適用"""
    df_smooth = df_input.copy()
    
    exclude_keywords = ['Time (Seconds)', 'Marker']
    filter_target_cols = [col for col in df_smooth.columns 
                          if col not in exclude_keywords and 'Marker' not in col]
    
    # 欠損値補完
    df_smooth[filter_target_cols] = df_smooth[filter_target_cols].interpolate(
        method='linear', limit_direction='both')
    df_smooth[filter_target_cols] = df_smooth[filter_target_cols].fillna(0)
    
    # フィルタ適用
    df_smooth[filter_target_cols] = apply_lowpass_filter(
        df_smooth[filter_target_cols].values, cutoff=cutoff, fs=fs, order=order)
    
    # 圧力データのクリッピング（負値を0に）
    pressure_cols = [col for col in filter_target_cols if 'kPa' in col]
    if pressure_cols:
        df_pressure = df_smooth[pressure_cols]
        df_smooth[pressure_cols] = df_pressure.mask(df_pressure < 0, 0)
    
    return df_smooth

### 6. 関節角度計算
座標軸置換後
- XY平面　水平面
- YZ平面　矢状面
- ZX平面　前額面

In [8]:
import numpy as np
import pandas as pd
from numpy import linalg as LA

# ============================================================
#  マーカーインデックス定義（0-indexed）
#
#  グローバル座標系: Z=上, X=右, Y=前
#  右足の「外側」= グローバル +X 方向
#  左足の「外側」= グローバル −X 方向
# ============================================================
RI  = 0   # 右 ASIS（腸骨前上棘）
RGT = 1   # 右 大転子
RK  = 2   # 右 膝外側
RM  = 3   # 右 外果
RT  = 4   # 右 足の甲
LI  = 5   # 左 ASIS
LGT = 6   # 左 大転子
LK  = 7   # 左 膝外側
LM  = 8   # 左 外果
LT  = 9   # 左 足の甲

JOINT_NO = 6  # 股(R/L)，膝(R/L)，足(R/L)


# ============================================================
#  ユーティリティ関数
# ============================================================

def _normalize(v: np.ndarray) -> np.ndarray:
    """単位ベクトル化（ゼロベクトル安全）"""
    n = LA.norm(v)
    return v / n if n > 1e-10 else v.copy()


def _mean_rotation_matrix(R_list: list) -> np.ndarray:
    """
    複数の回転行列を平均化（SVD 直交化で SO(3) に射影）

    単純な要素ごとの平均は回転行列にならないため，
    SVD を使って最近傍の回転行列に射影する．
    """
    R_mean = np.mean(np.array(R_list), axis=0)
    U, _, Vt = LA.svd(R_mean)
    R_avg = U @ Vt
    if LA.det(R_avg) < 0:   # 反射（det=−1）になった場合の修正
        U[:, -1] *= -1
        R_avg = U @ Vt
    return R_avg


# ============================================================
#  セグメント座標系の構築
#
#  セグメント CS の定義（各列がローカル軸のグローバル方向）:
#    列0 = e_lat  (外側, X軸)
#    列1 = e_ant  (前方, Y軸)
#    列2 = e_sup  (上方/近位, Z軸)
#
#  右手系: e_lat × e_ant = e_sup  (X × Y = Z)
# ============================================================

def _build_cs_long_axis(e_long_raw: np.ndarray,
                         e_ml_ref: np.ndarray) -> np.ndarray:
    """
    Z軸（上方向）主軸
    長軸（近位方向）と参照 ML 軸からセグメント座標系を構築
    骨盤・大腿・下腿に使用

    Parameters
    ----------
    e_long_raw : 近位方向の参照ベクトル（正規化不要）
    e_ml_ref   : 外側方向の参照ベクトル（正規化不要）

    Returns
    -------
    R : (3, 3)  列 = [e_lat, e_ant, e_sup]

    手順
    ----
    1. e_sup = normalize(e_long_raw)
    2. e_lat = Gram-Schmidt(e_ml_ref ⊥ e_sup)
    3. e_ant = e_sup × e_lat  （Z × X = Y，右手系）
    4. e_sup = e_lat × e_ant  （完全直交化）
    """
    e_sup = _normalize(e_long_raw)
    e_lat = e_ml_ref - np.dot(e_ml_ref, e_sup) * e_sup
    e_lat = _normalize(e_lat)
    e_ant = _normalize(np.cross(e_sup, e_lat))   # Z × X = Y
    e_sup = _normalize(np.cross(e_lat, e_ant))   # X × Y = Z（再構成）
    return np.column_stack([e_lat, e_ant, e_sup])


def _build_cs_ant_axis(e_ant_raw: np.ndarray,
                        e_sup_ref: np.ndarray) -> np.ndarray:
    """
    y方向(前方向)主軸
    前後軸（足の長軸）と参照上軸からセグメント座標系を構築
    足部専用（主軸が前後方向のため骨盤等とは別処理）

    Parameters
    ----------
    e_ant_raw  : 前方（外果→足の甲）の参照ベクトル（正規化不要）
    e_sup_ref  : 上方の参照ベクトル（下腿の e_sup など）

    Returns
    -------
    R : (3, 3)  列 = [e_lat, e_ant, e_sup]

    手順
    ----
    1. e_ant = normalize(e_ant_raw)
    2. e_sup = Gram-Schmidt(e_sup_ref ⊥ e_ant)
    3. e_lat = e_ant × e_sup  （Y × Z = X，右手系）
    4. e_sup = e_lat × e_ant  （完全直交化）
    """
    e_ant = _normalize(e_ant_raw)
    e_sup = e_sup_ref - np.dot(e_sup_ref, e_ant) * e_ant
    e_sup = _normalize(e_sup)
    e_lat = _normalize(np.cross(e_ant, e_sup))   # Y × Z = X
    e_sup = _normalize(np.cross(e_lat, e_ant))   # X × Y = Z（再構成）
    return np.column_stack([e_lat, e_ant, e_sup])


def build_all_segment_cs(pos: np.ndarray, side: str):
    """
    1フレームのマーカー位置から全セグメント座標系を構築
 
    Parameters
    ----------
    pos  : (10, 3)  グローバル座標（Z=上, X=右, Y=前）
    side : 'right' or 'left'
 
    ★ 左側の鏡映処理 ★
    --------------------
    左側は全マーカーの X 座標を符号反転（矢状面で鏡映）してから
    右側と全く同じ計算を行う．
 
    鏡映後の空間では:
      - 左 ASIS の X が正（＝仮想右側）
      - 左大転子・膝・外果・足の甲も +X 側に来る
      - 左外側方向 → 鏡映後は +X 方向（右側と同じ）
 
    これにより:
      - e_ant が必ず +Y を向く（e_sup × e_lat = +Z × +X = +Y）
      - 屈曲: 前方への回転 → 正（左右共通）
      - 外転: 正中から離れる動き → 正（左右共通）
      - 外旋: 外向きへの回転 → 正（左右共通）
 
    Returns
    -------
    R_pelvis, R_thigh, R_shank, R_foot : 各 (3, 3)
    """
 
    # ── 左側: X座標を鏡映して仮想右側として扱う ──────────────
    if side == 'left':
        pos = pos.copy()
        pos[:, 0] *= -1   # ← 全マーカーのX座標を反転
 
    # インデックス（right/left 共通）
    if side == 'right':
        idx = {'asis': RI, 'gt': RGT, 'knee': RK,
               'mall': RM, 'toe': RT, 'asis_c': LI}
    else:
        idx = {'asis': LI, 'gt': LGT, 'knee': LK,
               'mall': LM, 'toe': LT, 'asis_c': RI}
 
    p_asis   = pos[idx['asis']]
    p_asis_c = pos[idx['asis_c']]
    p_gt     = pos[idx['gt']]
    p_knee   = pos[idx['knee']]
    p_mall   = pos[idx['mall']]
    p_toe    = pos[idx['toe']]
 
    # ── 骨盤 ─────────────────────────────────────────────────
    # ML軸: 同側ASIS - 対側ASIS → +X方向（鏡映済みなので両側とも+X）
    e_ml_pelvis = _normalize(p_asis - p_asis_c)
    R_pelvis = _build_cs_long_axis(np.array([0.0, 0.0, 1.0]), e_ml_pelvis)
 
    # ── 大腿 ─────────────────────────────────────────────────
    R_thigh = _build_cs_long_axis(p_gt - p_knee, R_pelvis[:, 0])
 
    # ── 下腿 ─────────────────────────────────────────────────
    R_shank = _build_cs_long_axis(p_knee - p_mall, R_thigh[:, 0])
 
    # ── 足部 ─────────────────────────────────────────────────
    R_foot = _build_cs_ant_axis(p_toe - p_mall, R_shank[:, 2])
 
    return R_pelvis, R_thigh, R_shank, R_foot


# ============================================================
#  回転行列 → 関節角度
#
#  XYZ オイラー角分解:  R_rel = Rx(flex) @ Ry(abd) @ Rz(rot)
#
#    flex = arctan2(−R[1,2],  R[2,2])  ← e_lat(X) 軸周り = 屈曲/背屈
#    abd  = arcsin( R[0,2])             ← e_ant(Y) 軸周り = 外転/内転（浮動軸近似）
#    rot  = arctan2(−R[0,1],  R[0,0])  ← e_sup(Z) 軸周り = 外旋/内旋
#
#  符号規約（立位基準）:
#    屈曲(+) / 伸展(−)
#    外転(+) / 内転(−)
#    外旋(+) / 内旋(−)
#    足関節: 背屈(+) / 底屈(−)
#
#  ⚠️ 符号は必ず静的立位データで実測値と照合すること
#     もし逆転していた場合は SIGN_TABLE で補正する
# ============================================================

# 符号補正テーブル（+1=そのまま, −1=反転）
# データ確認後に必要があれば各軸を −1 に変更する
SIGN_TABLE = {
    'hip':   np.array([1.0,  1.0,  1.0]),
    'knee':  np.array([-1.0,  1.0,  1.0]),
    'ankle': np.array([1.0,  1.0,  1.0]),
}


def rotation_to_euler_xyz(R_rel: np.ndarray,
                           joint: str = 'knee') -> np.ndarray:
    """
    相対回転行列 R_rel = R_prox.T @ R_dist から XYZ オイラー角を抽出

    Parameters
    ----------
    R_rel  : (3, 3)
    joint  : 'hip', 'knee', 'ankle'（符号補正テーブルの参照用）

    Returns
    -------
    angles : (3,) [屈曲/背屈, 外転, 外旋]  degree
    """
    flex = np.degrees(np.arctan2(-R_rel[1, 2],  R_rel[2, 2]))
    abd  = np.degrees(np.arcsin(np.clip( R_rel[0, 2], -1.0, 1.0)))
    rot  = np.degrees(np.arctan2(-R_rel[0, 1],  R_rel[0, 0]))
    return np.array([flex, abd, rot]) * SIGN_TABLE[joint]


# ============================================================
#  メイン計算クラス
# ============================================================

class CalculateAngle:
    """
    セグメント座標系ベースの関節角度計算クラス

    グローバル: Z=上, X=右, Y=前
    セグメント CS: 各列 = [e_lat, e_ant, e_sup]
    関節角度: [屈曲/背屈, 外転, 外旋]（degree）
    """

    def angles_one_side(self, pos: np.ndarray, side: str) -> np.ndarray:
        """
        片側の股・膝・足関節角度を計算

        Returns
        -------
        out : (3, 3)
            [[股flex, 股abd, 股rot],
             [膝flex, 膝abd, 膝rot],
             [足flex, 足abd, 足rot]]
        """
        Rp, Rt, Rs, Rf = build_all_segment_cs(pos, side)
        return np.stack([
            rotation_to_euler_xyz(Rp.T @ Rt, 'hip'),
            rotation_to_euler_xyz(Rt.T @ Rs, 'knee'),
            rotation_to_euler_xyz(Rs.T @ Rf, 'ankle'),
        ], axis=0)

    def angles(self, pos: np.ndarray) -> np.ndarray:
        """
        1フレームの全関節角度を計算

        Parameters
        ----------
        pos : (10, 3)  グローバル位置

        Returns
        -------
        out : (6, 3)
            行順: [R股, L股, R膝, L膝, R足, L足]
            列順: [屈曲/背屈, 外転, 外旋]
        """
        right = self.angles_one_side(pos, 'right')
        left  = self.angles_one_side(pos, 'left')
        return np.array([
            right[0], left[0],   # 股関節 R/L
            right[1], left[1],   # 膝関節 R/L
            right[2], left[2],   # 足関節 R/L
        ])


# ============================================================
#  静的キャリブレーション（基準回転行列の計算）
# ============================================================

def compute_static_joint_matrices(pos_frames: np.ndarray,
                                   side: str) -> tuple:
    """
    静的区間フレームから基準関節回転行列を計算

    Parameters
    ----------
    pos_frames : (N, 10, 3)
    side       : 'right' or 'left'

    Returns
    -------
    (R_hip_ref, R_knee_ref, R_ankle_ref) : 各 (3, 3)
        静的立位時の各関節回転行列（フレーム間平均）
    """
    R_hips, R_knees, R_ankles = [], [], []
    for pos in pos_frames:
        Rp, Rt, Rs, Rf = build_all_segment_cs(pos, side)
        R_hips.append(Rp.T @ Rt)
        R_knees.append(Rt.T @ Rs)
        R_ankles.append(Rs.T @ Rf)
    return (
        _mean_rotation_matrix(R_hips),
        _mean_rotation_matrix(R_knees),
        _mean_rotation_matrix(R_ankles),
    )


# ============================================================
#  全フレーム角度計算
# ============================================================

def calculate_angles_from_positions(
    df: pd.DataFrame,
    target_cols: list,
    R_ref_right: tuple = None,
    R_ref_left:  tuple = None,
) -> np.ndarray:
    """
    データフレームから全フレームの関節角度を計算

    Parameters
    ----------
    df           : フレームデータ（各行 = 1フレーム）
    target_cols  : 位置カラム名リスト（30列: 10マーカー × XYZ）
    R_ref_right  : (R_hip_ref, R_knee_ref, R_ankle_ref)  右側基準行列
                   None の場合はキャリブレーションなし（絶対角度）
    R_ref_left   : 同上，左側

    Returns
    -------
    all_angles : (N, 18)
        列順: R股(×3), L股(×3), R膝(×3), L膝(×3), R足(×3), L足(×3)
        各(×3): 屈曲/背屈, 外転, 外旋
    """
    pos_flat = df[target_cols].values
    n_frames = pos_flat.shape[0]

    if n_frames == 0:
        return np.empty((0, 18))

    pos_3d = pos_flat.reshape(n_frames, 10, 3)
    calc   = CalculateAngle()
    all_angles = np.zeros((n_frames, JOINT_NO, 3))

    use_calib = (R_ref_right is not None) and (R_ref_left is not None)

    joint_keys = ['hip', 'knee', 'ankle']

    for f, pos in enumerate(pos_3d):
        if not use_calib:
            # キャリブレーションなし: グローバル座標での絶対角度
            all_angles[f] = calc.angles(pos)
        else:
            # キャリブレーションあり: 静的基準からの相対角度
            # R_rel = R_ref.T @ R_dyn
            for s_idx, (side, R_refs) in enumerate(
                [('right', R_ref_right), ('left', R_ref_left)]
            ):
                Rp, Rt, Rs, Rf = build_all_segment_cs(pos, side)
                R_dyns = [Rp.T @ Rt, Rt.T @ Rs, Rs.T @ Rf]

                for j_idx, (R_dyn, R_ref, jkey) in enumerate(
                    zip(R_dyns, R_refs, joint_keys)
                ):
                    # out_row: 0=R股, 1=L股, 2=R膝, 3=L膝, 4=R足, 5=L足
                    out_row = j_idx * 2 + s_idx
                    all_angles[f, out_row] = rotation_to_euler_xyz(
                        R_ref.T @ R_dyn, jkey
                    )

    return all_angles.reshape(n_frames, JOINT_NO * 3)


# ============================================================
#  メイン処理
# ============================================================

def process_mocap_data_target_calibration(df_target, df_ref):
    """
    Parameters
    ----------
    df_target : 解析対象データ (250 Hz)
    df_ref    : タイミング特定用データ（'Marker' 列あり）

    Returns
    -------
    df_result   : DataFrame（角度 + Time列）
    R_refs_dict : {'right': (R_hip, R_knee, R_ankle),
                   'left' : (R_hip, R_knee, R_ankle)}
    columns     : list[str]  角度カラム名
    """

    target_cols = []
    for i in range(1, 11):
        prefix = f"{i:03}"
        target_cols.extend([
            f"{prefix}_Position_X",
            f"{prefix}_Position_Y",
            f"{prefix}_Position_Z",
        ])

    # ── 1. キャリブレーション時間の特定 ──────────────────────
    print("キャリブレーション時間を特定中 (from df_ref)...")
    trigger_rows = df_ref[df_ref['Marker'] == 2]
    if trigger_rows.empty:
        raise ValueError("df_ref に Marker == 2 のデータが見つかりません．")

    trigger_time = trigger_rows.iloc[0]['Time (Seconds)']
    start_time   = trigger_time - 20.0
    end_time     = trigger_time - 15.0
    print(f"トリガー検知: {trigger_time:.4f} sec")
    print(f"静的基準区間: {start_time:.2f} 〜 {end_time:.2f} sec")

    # ── 2. 静的区間データの抽出 ──────────────────────────────
    mask = (
        (df_target['Time (Seconds)'] >= start_time) &
        (df_target['Time (Seconds)'] <= end_time)
    )
    df_base = df_target.loc[mask].copy()
    print(f"静的区間フレーム数 (df_target): {len(df_base)} frames")

    if len(df_base) == 0:
        raise ValueError(
            f"df_target 内に指定時間 ({start_time:.2f}〜{end_time:.2f} sec) "
            "のデータが存在しません．"
        )

    # ── 3. 基準関節回転行列の計算 ────────────────────────────
    print("静的基準回転行列を計算中...")
    pos_base = df_base[target_cols].values.reshape(-1, 10, 3)

    R_ref_right = compute_static_joint_matrices(pos_base, 'right')
    R_ref_left  = compute_static_joint_matrices(pos_base, 'left')

    # 検証: 基準区間での相対角度は ≈ 0° になるはず
    static_angles = calculate_angles_from_positions(
        df_base, target_cols, R_ref_right, R_ref_left
    )
    print("\n【静的基準区間での角度（≈ 0° になれば正常）】")
    tmp_cols = []
    for j in ["R_Hip", "L_Hip", "R_Knee", "L_Knee", "R_Ankle", "L_Ankle"]:
        for d in ["Flex", "Abd", "Rot"]:
            tmp_cols.append(f"{j}_{d}")
    df_check = pd.DataFrame(static_angles, columns=tmp_cols)
    print(df_check.mean().round(3).to_string())
    print()

    # ── 4. 全フレームの角度計算 ──────────────────────────────
    print("全フレームの角度計算中...")
    angles_all = calculate_angles_from_positions(
        df_target, target_cols,
        R_ref_right=R_ref_right,
        R_ref_left=R_ref_left,
    )

    # ── 5. DataFrame 作成 ────────────────────────────────────
    joint_names = [
        "Right_Hip", "Left_Hip",
        "Right_Knee", "Left_Knee",
        "Right_Ankle", "Left_Ankle",
    ]
    dof_names = ["Flex_Ext", "Abd_Add", "Int_Ext_Rot"]
    columns = [f"{j}_{d}" for j in joint_names for d in dof_names]

    df_result = pd.DataFrame(angles_all, columns=columns)
    if 'Time (Seconds)' in df_target.columns:
        df_result.insert(0, 'Time (Seconds)', df_target['Time (Seconds)'].values)

    R_refs_dict = {'right': R_ref_right, 'left': R_ref_left}
    return df_result, R_refs_dict, columns

### 7. 床反力の正規化（体重比）[%BW]

In [9]:
def normalize_force_by_bodyweight(df_force, mass):
    """床反力を体重で正規化"""
    columns = ['Time (Seconds)', 'Right_Fx', 'Right_Fy', 'Right_Fz', 
               'Left_Fx', 'Left_Fy', 'Left_Fz']
    force_columns = ['Right_Fx', 'Right_Fy', 'Right_Fz', 'Left_Fx', 'Left_Fy', 'Left_Fz']
    
    body_weight = mass * 9.81
    df_normalized = df_force[columns].copy()
    df_normalized[force_columns] /= body_weight
    
    return df_normalized

### 8. 正規化

In [10]:
def process_hybrid_normalization(df_input, duration=300):
    """
    マーカー2から指定時間のデータを使用し、
    - 圧力データ (Pressure) -> Min-Max (0~1)
    - IMUデータ (その他)    -> Z-score (平均0, 分散1)
    に正規化する
    """
    df_norm = df_input.copy()
    df_norm.columns = df_norm.columns.str.replace('kPa', 'Pressure')
    
    exclude_keywords = ['Time (Seconds)', 'Marker']

    # ---------------------------------------------------------
    # 1. 列の分類 (圧力か、それ以外(IMU)か)
    # ---------------------------------------------------------
    # 'Pressure' という文字が含まれる列を抽出
    pressure_cols = [col for col in df_norm.columns if 'Pressure' in col]
    
    # それ以外の数値データ列をIMUとして扱う
    imu_cols = [col for col in df_norm.columns 
                if col not in exclude_keywords 
                and 'Marker' not in col 
                and col not in pressure_cols]
    
    # ---------------------------------------------------------
    # 2. 基準となる期間(300秒)のデータ抽出
    # ---------------------------------------------------------
    start_time = 0
    marker_cols = [col for col in df_norm.columns if 'Marker' in col]
    
    if marker_cols:
        marker_2_rows = df_norm[(df_norm[marker_cols] == 2).any(axis=1)]
        if not marker_2_rows.empty:
            start_time = marker_2_rows.iloc[0]['Time (Seconds)']
    
    end_time = start_time + duration
    mask = (df_norm['Time (Seconds)'] >= start_time) & (df_norm['Time (Seconds)'] <= end_time)
    df_stats_base = df_norm.loc[mask]
    
    # ---------------------------------------------------------
    # 3. IMUデータの正規化 -> Z-score
    # ---------------------------------------------------------
    if imu_cols:
        means = df_stats_base[imu_cols].mean()
        stds = df_stats_base[imu_cols].std()
        stds = stds.replace(0, 1) # 0除算回避
        
        # 全データに対して適用
        df_norm[imu_cols] = (df_norm[imu_cols] - means) / stds

    # ---------------------------------------------------------
    # 4. 足底圧力データの正規化 -> Min-Max (0~1)
    # ---------------------------------------------------------
    if pressure_cols:
        mins = df_stats_base[pressure_cols].min()
        maxs = df_stats_base[pressure_cols].max()
        ranges = maxs - mins
        ranges = ranges.replace(0, 1) # 0除算回避
        
        # 全データに対して適用
        df_norm[pressure_cols] = (df_norm[pressure_cols] - mins) / ranges

    # 戻り値: データフレームと、後で確認・逆変換できるように統計量も返す
    stats_info = {
        "imu_mean": means if imu_cols else None,
        "imu_std": stds if imu_cols else None,
        "press_min": mins if pressure_cols else None,
        "press_max": maxs if pressure_cols else None
    }
    
    return df_norm, stats_info

def process_zscore_normalization(df_input, duration=300):
    """マーカー2から指定時間のデータでZスコア化"""
    df_z = df_input.copy()
    df_z.columns = df_z.columns.str.replace('kPa', 'Pressure')
    
    exclude_keywords = ['Time (Seconds)', 'Marker']
    target_cols = [col for col in df_z.columns 
                   if col not in exclude_keywords and 'Marker' not in col]
    
    marker_cols = [col for col in df_z.columns if 'Marker' in col]
    start_time = 0
    
    if marker_cols:
        marker_2_rows = df_z[(df_z[marker_cols] == 2).any(axis=1)]
        if not marker_2_rows.empty:
            start_time = marker_2_rows.iloc[0]['Time (Seconds)']
    
    end_time = start_time + duration
    mask = (df_z['Time (Seconds)'] >= start_time) & (df_z['Time (Seconds)'] <= end_time)
    df_stats_base = df_z.loc[mask, target_cols]
    
    means = df_stats_base.mean()
    stds = df_stats_base.std()
    stds = stds.replace(0, 1)
    
    df_z[target_cols] = (df_z[target_cols] - means) / stds
    
    return df_z, means, stds

### 9. データ同期と結合

In [11]:
def calculate_fine_offset_pressure(df_target, df_ref, col_target_pressure_list, 
                                   col_ref_name, t_start, duration=300, fs=100):
    """足底圧と床反力の相互相関によるタイミング補正"""
    t_end = t_start + duration
    
    mask_tgt = (df_target['Time (Seconds)'] >= t_start) & (df_target['Time (Seconds)'] <= t_end)
    df_t = df_target.loc[mask_tgt].copy()

    mask_ref = (df_ref['Time (Seconds)'] >= t_start) & (df_ref['Time (Seconds)'] <= t_end)
    df_r = df_ref.loc[mask_ref].copy()
    
    if len(df_t) < fs or len(df_r) < fs:
        return 0.0
    
    t_min = max(df_t['Time (Seconds)'].min(), df_r['Time (Seconds)'].min())
    t_max = min(df_t['Time (Seconds)'].max(), df_r['Time (Seconds)'].max())
    
    if t_min >= t_max:
        return 0.0
    
    common_t = np.arange(t_min, t_max, 1.0/fs)
    
    pressure_sum = df_t[col_target_pressure_list].sum(axis=1)
    f_tgt = interp1d(df_t['Time (Seconds)'], pressure_sum, 
                     kind='linear', fill_value=0, bounds_error=False)
    sig_tgt = f_tgt(common_t)
    
    f_ref = interp1d(df_r['Time (Seconds)'], df_r[col_ref_name], 
                     kind='linear', fill_value=0, bounds_error=False)
    sig_ref = f_ref(common_t)
    
    sig_tgt_norm = (sig_tgt - np.mean(sig_tgt)) / (np.std(sig_tgt) + 1e-6)
    sig_ref_norm = (sig_ref - np.mean(sig_ref)) / (np.std(sig_ref) + 1e-6)
    
    correlation = signal.correlate(sig_ref_norm, sig_tgt_norm, mode='full')
    lags = signal.correlation_lags(len(sig_ref_norm), len(sig_tgt_norm), mode='full')
    best_lag = lags[np.argmax(correlation)]
    
    return best_lag / fs

def synchronize_merge_and_extract(df_left, df_right, df_angles, df_force, target_freq=100):
    """
    全データの同期・結合・抽出
    Marker 1で粗調整 → 足底圧で微調整 → 結合 → Marker 2から300秒抽出
    """
    # Marker 1による粗調整
    trigger_marker = 1
    marker_rows_l = df_left[df_left['Marker'] == trigger_marker]
    t_marker_left_1 = 0.0
    if not marker_rows_l.empty:
        t_marker_left_1 = marker_rows_l.iloc[0]['Time (Seconds)']
    
    df_right_rough = df_right.copy()
    marker_rows_r = df_right[df_right['Marker'] == trigger_marker]
    if not marker_rows_r.empty:
        t_marker_r_1 = marker_rows_r.iloc[0]['Time (Seconds)']
        offset_r_rough = t_marker_left_1 - t_marker_r_1
        df_right_rough['Time (Seconds)'] += offset_r_rough
    
    df_force_rough = df_force.copy()
    df_force_rough['Time (Seconds)'] += t_marker_left_1
    
    # Marker 2 + 300sによる微調整
    fine_tune_marker = 2
    duration = 300
    cols_Pressure_left = [f'Left_Pressure_{i}' for i in range(1, 9)]
    cols_Pressure_right = [f'Right_Pressure_{i}' for i in range(1, 9)]
    
    marker_rows_l_2 = df_left[df_left['Marker'] == fine_tune_marker]
    
    df_left_final = df_left.copy()
    df_right_final = df_right_rough.copy()
    df_angles_final = df_angles.copy()
    
    if not marker_rows_l_2.empty:
        t_start_fine = marker_rows_l_2.iloc[0]['Time (Seconds)']
        
        offset_l_fine = calculate_fine_offset_pressure(
            df_target=df_left, df_ref=df_force_rough, 
            col_target_pressure_list=cols_Pressure_left,
            col_ref_name='Left_Fz', t_start=t_start_fine, 
            duration=duration, fs=target_freq
        )
        
        offset_r_fine = calculate_fine_offset_pressure(
            df_target=df_right_rough, df_ref=df_force_rough, 
            col_target_pressure_list=cols_Pressure_right,
            col_ref_name='Right_Fz', t_start=t_start_fine, 
            duration=duration, fs=target_freq
        )
        
        df_left_final['Time (Seconds)'] += offset_l_fine
        df_angles_final['Time (Seconds)'] += offset_l_fine
        df_right_final['Time (Seconds)'] += offset_r_fine
    
    df_force_final = df_force_rough
    
    # 統合用時間軸の作成
    t_start = max(df_left_final['Time (Seconds)'].min(), 
                  df_right_final['Time (Seconds)'].min(),
                  df_angles_final['Time (Seconds)'].min(), 
                  df_force_final['Time (Seconds)'].min())
    t_end = min(df_left_final['Time (Seconds)'].max(), 
                df_right_final['Time (Seconds)'].max(),
                df_angles_final['Time (Seconds)'].max(), 
                df_force_final['Time (Seconds)'].max())
    
    common_time = np.arange(t_start, t_end, 1.0/target_freq)
    df_merged = pd.DataFrame({'Time (Seconds)': common_time})
    
    # リサンプリングと結合
    data_sources = {'L_Dev': df_left_final, 'R_Dev': df_right_final, 
                    'Mocap': df_angles_final, 'Force': df_force_final}
    
    for prefix, df_src in data_sources.items():
        time_col = 'Time (Seconds)' if 'Time (Seconds)' in df_src.columns else 'Time'
        numeric_cols = df_src.select_dtypes(include=[np.number]).columns
        cols_to_interp = [c for c in numeric_cols if c != time_col and 'Marker' not in c]
        
        if not cols_to_interp:
            continue
        
        f = interp1d(df_src[time_col], df_src[cols_to_interp], axis=0, 
                     kind='linear', fill_value="extrapolate")
        interp_data = f(common_time)
        df_temp = pd.DataFrame(interp_data, columns=cols_to_interp)
        df_merged = pd.concat([df_merged, df_temp], axis=1)
    
    df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]
    
    # Marker 2から300秒間の抽出
    marker_rows_sync = df_left_final[df_left_final['Marker'] == 2]
    
    if not marker_rows_sync.empty:
        synced_start_time = marker_rows_sync.iloc[0]['Time (Seconds)']
        synced_end_time = synced_start_time + 300.0
        
        df_analysis = df_merged[
            (df_merged['Time (Seconds)'] >= synced_start_time) & 
            (df_merged['Time (Seconds)'] <= synced_end_time)
        ].copy()
        
        df_analysis['Time (Seconds)'] = df_analysis['Time (Seconds)'] - synced_start_time
        df_analysis = df_analysis.reset_index(drop=True)
        
        return df_analysis
    else:
        return df_merged

### 10. ストライド検出と抽出

In [12]:
def detect_fz_heel_strikes(signal_array, threshold=0.05, min_dist_samples=40):
    """Fzの立ち上がり検出"""
    is_contact = signal_array > threshold
    rising_edge = np.diff(is_contact.astype(int), prepend=0) == 1
    potential_indices = np.where(rising_edge)[0]
    
    if len(potential_indices) == 0:
        return np.array([])
    
    true_indices = [potential_indices[0]]
    for idx in potential_indices[1:]:
        if idx - true_indices[-1] > min_dist_samples:
            true_indices.append(idx)
    
    return np.array(true_indices)

def slice_strides_with_constraints(df_input, target_col, side_name="Left", 
                                   threshold=0.05, fs=100, 
                                   min_duration=0.7, max_duration=1.8):
    """ストライド時間制約を満たす歩行のみ抽出"""
    signal = df_input[target_col].values
    time_array = df_input['Time (Seconds)'].values
    
    min_dist_samples = int(0.4 * fs)
    hs_indices = detect_fz_heel_strikes(signal, threshold=threshold, 
                                        min_dist_samples=min_dist_samples)
    
    valid_strides = []
    
    for i in range(1, len(hs_indices) - 2):
        start_idx = hs_indices[i]
        end_idx = hs_indices[i+1]
        
        start_t = time_array[start_idx]
        end_t = time_array[end_idx]
        duration = end_t - start_t
        
        if min_duration <= duration <= max_duration:
            stride_df = df_input.iloc[start_idx:end_idx].copy()
            valid_strides.append(stride_df)
    
    print(f"[{side_name}] Accepted Strides: {len(valid_strides)}")
    return valid_strides


### 11. ストライドの正規化

In [13]:
def normalize_strides(stride_list, target_cols, n_points=200):
    """各ストライドを0-100%に正規化"""
    normalized_dfs = []
    data_collector = []
    
    gait_cycle = np.linspace(0, 100, n_points)
    x_new = np.linspace(0, 1, n_points)
    
    for stride_df in stride_list:
        n_len = len(stride_df)
        x_old = np.linspace(0, 1, n_len)
        
        new_df = pd.DataFrame()
        new_df['Gait Cycle (%)'] = gait_cycle
        
        stride_matrix = []
        
        for col in target_cols:
            if col in stride_df.columns:
                y_old = stride_df[col].values
                f = interp1d(x_old, y_old, kind='linear', fill_value="extrapolate")
                y_new = f(x_new)
                new_df[col] = y_new
                stride_matrix.append(y_new)
            else:
                zeros = np.zeros(n_points)
                new_df[col] = zeros
                stride_matrix.append(zeros)
        
        normalized_dfs.append(new_df)
        data_collector.append(np.array(stride_matrix).T)
    
    if len(data_collector) > 0:
        ensemble_array = np.array(data_collector)
    else:
        ensemble_array = np.empty((0, n_points, len(target_cols)))
    
    return normalized_dfs, ensemble_array

### 12. 外れ値ストライドの除去

In [14]:
def filter_outlier_strides(ensemble_array, stride_dfs, n_sigmas=3, 
                           outlier_ratio_threshold=0.05):
    """集団から大きく乖離した外れ値ストライドを除外"""
    median_curve = np.median(ensemble_array, axis=0)
    std_curve = np.std(ensemble_array, axis=0)
    
    upper_limit = median_curve + (n_sigmas * std_curve)
    lower_limit = median_curve - (n_sigmas * std_curve)
    
    upper_limit_bc = upper_limit[np.newaxis, :, :]
    lower_limit_bc = lower_limit[np.newaxis, :, :]
    
    is_outlier_matrix = (ensemble_array > upper_limit_bc) | (ensemble_array < lower_limit_bc)
    
    total_points_per_stride = ensemble_array.shape[1] * ensemble_array.shape[2]
    outlier_counts = np.sum(is_outlier_matrix, axis=(1, 2))
    outlier_ratios = outlier_counts / total_points_per_stride
    
    keep_mask = outlier_ratios <= outlier_ratio_threshold
    
    clean_ensemble = ensemble_array[keep_mask]
    clean_dfs = [df for i, df in enumerate(stride_dfs) if keep_mask[i]]
    
    return clean_ensemble, clean_dfs, keep_mask

 ### 13. 左右データの統合処理

In [15]:
def merge_left_right_data(left_ensemble, right_ensemble, left_cols, right_cols):
    """
    左右のデータを統合し，左足データを適切に反転して同じデータとして扱う
    
    【反転処理の詳細】
    - 加速度: Left_Accel_X を -1倍
    - 角速度: Left_Gyro_Y, Left_Gyro_Z を -1倍
    - 床反力: Left_Fx を -1倍
    
    Returns:
        merged_ensemble: 統合された3次元配列 (左右合計ストライド数, 200, 特徴量数)
        merged_cols: 統合されたカラム名リスト（Left_/Right_プレフィックスを削除）
    """
    # 右足データはそのまま使用
    right_data = right_ensemble.copy()
    
    # 左足データをコピーして反転処理
    left_data = left_ensemble.copy()
    
    print("\n--- 左足データの反転処理 ---")
    
    # 1. 加速度X軸の反転
    if 'Left_Accel_X' in left_cols:
        idx = left_cols.index('Left_Accel_X')
        left_data[:, :, idx] *= -1
        print(f"  ✓ Left_Accel_X を反転 (index: {idx})")
    
    # 2. 角速度Y, Z軸の反転
    for axis in ['Y', 'Z']:
        col_name = f'Left_Gyro_{axis}'
        if col_name in left_cols:
            idx = left_cols.index(col_name)
            left_data[:, :, idx] *= -1
            print(f"  ✓ Left_Gyro_{axis} を反転 (index: {idx})")
    
    # # 3. 関節角度ZX平面の反転
    # for joint in ['Hip', 'Knee', 'Ankle']:
    #     col_name = f'Left_{joint}_ZX'
    #     if col_name in left_cols:
    #         idx = left_cols.index(col_name)
    #         left_data[:, :, idx] *= -1
    #         print(f"  ✓ Left_{joint}_ZX を反転 (index: {idx})")
    
    # 4. 床反力Fxの反転
    if 'Left_Fx' in left_cols:
        idx = left_cols.index('Left_Fx')
        left_data[:, :, idx] *= -1
        print(f"  ✓ Left_Fx を反転 (index: {idx})")
    
    # 左右データの結合
    merged_ensemble = np.concatenate([left_data, right_data], axis=0)
    
    # カラム名は左右で共通化（Left_/Right_プレフィックスを削除）
    # 例: Left_Accel_X, Right_Accel_X → Accel_X
    merged_cols = [col.replace('Left_', '').replace('Right_', '') for col in left_cols]
    
    print(f"\n統合完了:")
    print(f"  左足ストライド数: {left_ensemble.shape[0]}")
    print(f"  右足ストライド数: {right_ensemble.shape[0]}")
    print(f"  合計ストライド数: {merged_ensemble.shape[0]}")
    print(f"  特徴量数: {merged_ensemble.shape[2]}")
    
    return merged_ensemble, merged_cols

### 14. メイン処理パイプライン

In [16]:
def process_single_participant_condition(participant, condition, mass):
    """1人の1条件分のデータを処理"""
    
    # データ読み込み
    data = load_participant_data(participant, condition, mass)
    
    # 列名整理
    df_mocap = clean_mocap_columns(data['mocap'])
    df_force = clean_force_columns(data['force'])
    
    # リサンプリング
    df_left_processed = process_resampling(data['left'], sampling_interval=10)
    df_right_processed = process_resampling(data['right'], sampling_interval=10)
    
    # ローパスフィルタ
    df_left_smoothed = process_smoothing_dataframe(df_left_processed, fs=DEVICE_FREQ)
    df_right_smoothed = process_smoothing_dataframe(df_right_processed, fs=DEVICE_FREQ)
    df_mocap_smoothed = process_smoothing_dataframe(df_mocap, fs=MOCAP_FREQ)
    df_force_smoothed = process_smoothing_dataframe(df_force, fs=FORCE_FREQ)
    
    # 関節角度計算
    df_angles, _, _ = process_mocap_data_target_calibration(
        df_mocap_smoothed, df_left_smoothed)
    
    # 床反力の正規化
    df_normalized = normalize_force_by_bodyweight(df_force_smoothed, mass)
    
    # Zスコア正規化
    df_left_z, _ = process_hybrid_normalization(df_left_smoothed, duration=300)
    df_right_z, _ = process_hybrid_normalization(df_right_smoothed, duration=300)
    
    # データ同期と結合
    df_final = synchronize_merge_and_extract(
        df_left_z, df_right_z, df_angles, df_normalized, target_freq=100)
    
    # ストライド抽出
    left_strides = slice_strides_with_constraints(
        df_final, 'Left_Fz', side_name="Left", threshold=0.05)
    right_strides = slice_strides_with_constraints(
        df_final, 'Right_Fz', side_name="Right", threshold=0.05)
    
    # カラム定義
    cols_left = [
        'Left_Pressure_1', 'Left_Pressure_2', 'Left_Pressure_3', 'Left_Pressure_4', 
        'Left_Pressure_5', 'Left_Pressure_6', 'Left_Pressure_7', 'Left_Pressure_8',
        'Left_Accel_X', 'Left_Accel_Y', 'Left_Accel_Z', 
        'Left_Gyro_X', 'Left_Gyro_Y', 'Left_Gyro_Z', 
        'Left_Hip_Flex_Ext', 'Left_Hip_Abd_Add', 'Left_Hip_Int_Ext_Rot',
        'Left_Knee_Flex_Ext', 'Left_Knee_Abd_Add', 'Left_Knee_Int_Ext_Rot',
        'Left_Ankle_Flex_Ext', 'Left_Ankle_Abd_Add', 'Left_Ankle_Int_Ext_Rot',
        'Left_Fx', 'Left_Fy', 'Left_Fz'
    ]
    cols_right = [c.replace('Left', 'Right') for c in cols_left]
    
    # 正規化
    left_norm_dfs, left_ensemble = normalize_strides(left_strides, cols_left, n_points=200)
    right_norm_dfs, right_ensemble = normalize_strides(right_strides, cols_right, n_points=200)
    
    # 外れ値除去
    L_ens_clean, L_dfs_clean, _ = filter_outlier_strides(
        left_ensemble, left_norm_dfs, n_sigmas=3, outlier_ratio_threshold=0.01)
    R_ens_clean, R_dfs_clean, _ = filter_outlier_strides(
        right_ensemble, right_norm_dfs, n_sigmas=3, outlier_ratio_threshold=0.01)
    
    # 左右データの統合（反転処理を含む）
    merged_ensemble, merged_cols = merge_left_right_data(
        L_ens_clean, R_ens_clean, cols_left, cols_right)
    
    return {
        'participant': participant,
        'condition': condition,
        'mass': mass,
        'ensemble': merged_ensemble,
        'columns': merged_cols,
        'left_dfs': L_dfs_clean,
        'right_dfs': R_dfs_clean
    }


### 15. 全実験協力者・全条件の一括処理

In [17]:
def process_all_data():
    """全実験協力者・全条件のデータを処理"""
    all_results = []
    
    for i, participant in enumerate(PARTICIPANTS):
        mass = MASSES[i]
        
        for condition in CONDITIONS:
            try:
                result = process_single_participant_condition(participant, condition, mass)
                all_results.append(result)
                
                print(f"\n✓ 完了: {participant}_{condition}")
                print(f"  統合ストライド数: {result['ensemble'].shape[0]}")
                print(f"  特徴量数: {result['ensemble'].shape[2]}")
                
            except Exception as e:
                print(f"\n✗ エラー: {participant}_{condition}")
                print(f"  {str(e)}")
                continue
    
    return all_results

### 16. データセットの保存

In [18]:
def save_dataset(all_results, output_dir='new_processed_data'):
    """処理済みデータをnpz形式で保存"""
    os.makedirs(output_dir, exist_ok=True)

    # 1. 被験者名とIDの対応表を作成 (例: {'oba': 0, 'ono': 1, ...})
    # all_resultsからユニークな名前を取得してソートすることでIDを固定
    unique_participants = sorted(list(set(r['participant'] for r in all_results)))
    participant_map = {name: i for i, name in enumerate(unique_participants)}
    
    print(f"ID Map: {participant_map}")

    # 統合用リスト
    all_ensembles = []
    all_subject_ids = []  # ★ここに追加: IDを格納するリスト
    
    # 個別ファイルとして保存
    for result in all_results:
        filename = f"{result['participant']}_{result['condition']}_processed.npz"
        filepath = os.path.join(output_dir, filename)
        
        np.savez(
            filepath,
            ensemble=result['ensemble'],
            columns=result['columns'],
            participant=result['participant'],
            condition=result['condition'],
            mass=result['mass']
        )
        print(f"保存完了: {filepath}")
        
        # 統合用データの準備
        data = result['ensemble']
        name = result['participant']
        
        # ★ここが重要: データ数と同じ長さのID配列を作成
        # 例: データが337個でIDが0なら、[0, 0, ..., 0] (長さ337) を作る
        n_samples = data.shape[0]
        current_id = participant_map[name]
        ids = np.full(n_samples, current_id)
        
        all_ensembles.append(data)
        all_subject_ids.append(ids)
    
    # 全データを統合
    combined_ensemble = np.concatenate(all_ensembles, axis=0) # (N, 200, 26)
    combined_ids = np.concatenate(all_subject_ids, axis=0)    # (N, )
    
    # 保存
    combined_path = os.path.join(output_dir, 'all_data_combined.npz')
    np.savez(
        combined_path,
        ensemble=combined_ensemble,
        subject_ids=combined_ids,      # ★追加: 被験者ID配列
        columns=all_results[0]['columns'],
        id_map=participant_map         # 参考: IDと名前の対応表も保存しておくと便利
    )
    
    print(f"\\n統合データ保存完了: {combined_path}")
    print(f"総ストライド数 (ensemble): {combined_ensemble.shape}")
    print(f"総ID数 (subject_ids): {combined_ids.shape}")

### 実行

In [19]:
if __name__ == "__main__":
    # 全データ処理
    all_results = process_all_data()
    
    # データセット保存
    save_dataset(all_results)
    
    print("\n" + "="*60)
    print("全処理が完了しました")
    print("="*60)


Processing: oba - Condition: H (Mass: 60.9 kg)
キャリブレーション時間を特定中 (from df_ref)...
トリガー検知: 37.3400 sec
静的基準区間: 17.34 〜 22.34 sec
静的区間フレーム数 (df_target): 1250 frames
静的基準回転行列を計算中...

【静的基準区間での角度（≈ 0° になれば正常）】
R_Hip_Flex      0.0
R_Hip_Abd       0.0
R_Hip_Rot       0.0
L_Hip_Flex      0.0
L_Hip_Abd      -0.0
L_Hip_Rot      -0.0
R_Knee_Flex     0.0
R_Knee_Abd      0.0
R_Knee_Rot     -0.0
L_Knee_Flex    -0.0
L_Knee_Abd      0.0
L_Knee_Rot      0.0
R_Ankle_Flex   -0.0
R_Ankle_Abd     0.0
R_Ankle_Rot    -0.0
L_Ankle_Flex   -0.0
L_Ankle_Abd     0.0
L_Ankle_Rot    -0.0

全フレームの角度計算中...
[Left] Accepted Strides: 230
[Right] Accepted Strides: 208

--- 左足データの反転処理 ---
  ✓ Left_Accel_X を反転 (index: 8)
  ✓ Left_Gyro_Y を反転 (index: 12)
  ✓ Left_Gyro_Z を反転 (index: 13)
  ✓ Left_Fx を反転 (index: 23)

統合完了:
  左足ストライド数: 195
  右足ストライド数: 141
  合計ストライド数: 336
  特徴量数: 26

✓ 完了: oba_h
  統合ストライド数: 336
  特徴量数: 26

Processing: oba - Condition: M (Mass: 60.9 kg)
キャリブレーション時間を特定中 (from df_ref)...
トリガー検知: 35.6700 sec
静的基準区間: 1